In [2]:
# ==========================================================
# Notebook 07: RAG Evaluation and Debugging
# ==========================================================

import faiss
import pickle
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer, util

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
chunks_df = pd.read_csv("data/processed/document_chunks.csv")

faiss_index = faiss.read_index("data/embeddings/faiss_index.bin")

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
evaluation_data = [
    {
        "question": "What was the company's revenue?",
        "ground_truth": "The company's revenue was $120 million.",
    },
    {
        "question": "What was the net profit?",
        "ground_truth": "The net profit was $30 million.",
    },
    {
        "question": "What was the CEO salary?",
        "ground_truth": "The information was not found in the document.",
    },
]

evaluation_df = pd.DataFrame(evaluation_data)

evaluation_df

,question,ground_truth
0,What was the company's revenue?,The company's revenue was $120 million.
1,What was the net profit?,The net profit was $30 million.
2,What was the CEO salary?,The information was not found in the document.


In [5]:
def inspect_retrieval(query, retrieved_docs):

    print("=" * 80)
    print("QUESTION:")
    print(query)

    print("\nRETRIEVED CHUNKS:\n")

    for idx, row in retrieved_docs.iterrows():

        print("-" * 80)

        print(f"Page: {row['page_number']}")

        print(f"Similarity: {row['similarity_score']:.3f}")

        print(row["chunk_text"][:250])

In [20]:
# ==========================================================
# Grounded RAG Retrieval Function
# ==========================================================


def grounded_rag(
    query,
    top_k=5,
):

    # Create query embedding
    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
    )

    query_embedding = np.array(
        [query_embedding],
        dtype="float32",
    )

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search FAISS index
    scores, indices = faiss_index.search(
        query_embedding,
        top_k,
    )

    retrieved_records = []

    for score, idx in zip(
        scores[0],
        indices[0],
    ):

        row = chunks_df.iloc[idx]

        retrieved_records.append(
            {
                "chunk_id": row["chunk_id"],
                "source_document": row["source_document"],
                "page_number": row["page_number"],
                "chunk_text": row["chunk_text"],
                "similarity_score": float(score),
            }
        )

    retrieved_docs = pd.DataFrame(retrieved_records)

    # Build context string
    context = "\n\n".join(retrieved_docs["chunk_text"].tolist())

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
    }

In [21]:
result = grounded_rag(query="What was the company's revenue?")

inspect_retrieval(query=result["query"], retrieved_docs=result["retrieved_docs"])

QUESTION:
What was the company's revenue?

RETRIEVED CHUNKS:

--------------------------------------------------------------------------------
Page: 102
Similarity: 0.568
. Opening Stock Finished goods 108.21 126.67 (c) Reconciling the amount of revenue recognised in the Statement of Profit and Loss with the contracted price : Stock-in-trade 375.94 367.69 For the year ended For the year ended Work-in-progress 17.76 15
--------------------------------------------------------------------------------
Page: 102
Similarity: 0.538
- Sales Return (154.11) (168.31) Work-in-progress (20.43) (17.76) - Discounts (36.64) (26.35) (180.68) 7.73 Net revenue from sale of products and rendering of services 6,345.40 5,779.83 28 EMPLOYEE BENEFITS EXPENSE Information about the Company’s per
--------------------------------------------------------------------------------
Page: 4
Similarity: 0.504
. $ 42 Billion WORLDWIDE SALES The Abbott we Have Built 9.6%+ Our diversified business model proved its power a

In [22]:
def answer_relevance(question, answer):

    embeddings = embedding_model.encode([question, answer], convert_to_tensor=True)

    score = util.cos_sim(embeddings[0], embeddings[1])

    return float(score)

In [23]:
score = answer_relevance(
    question="What was the company's revenue?",
    answer="The company's revenue was $120 million.",
)

print("Answer Relevance:", round(score, 3))

Answer Relevance: 0.793


In [24]:
def context_precision(question, retrieved_docs):

    question_embedding = embedding_model.encode(question, convert_to_tensor=True)

    scores = []

    for chunk in retrieved_docs["chunk_text"]:

        chunk_embedding = embedding_model.encode(chunk, convert_to_tensor=True)

        similarity = util.cos_sim(question_embedding, chunk_embedding)

        scores.append(float(similarity))

    return np.mean(scores)

In [25]:
context_score = context_precision(
    question=result["query"], retrieved_docs=result["retrieved_docs"]
)

print("Context Precision:", round(context_score, 3))

Context Precision: 0.522


In [26]:
def faithfulness(answer, retrieved_docs):

    context = " ".join(retrieved_docs["chunk_text"].tolist())

    embeddings = embedding_model.encode([answer, context], convert_to_tensor=True)

    score = util.cos_sim(embeddings[0], embeddings[1])

    return float(score)

In [27]:
# Simulate answer using top retrieved chunk
generated_answer = result["retrieved_docs"].iloc[0]["chunk_text"]

faithfulness_score = faithfulness(
    answer=generated_answer,
    retrieved_docs=result["retrieved_docs"],
)

print(
    "Faithfulness:",
    round(faithfulness_score, 3),
)

Faithfulness: 0.841


In [30]:
def evaluate_rag(query):

    result = grounded_rag(query)

    # Use top retrieved chunk as pseudo-answer
    generated_answer = result["retrieved_docs"].iloc[0]["chunk_text"]

    answer_score = answer_relevance(
        query,
        generated_answer,
    )

    context_score = context_precision(
        query,
        result["retrieved_docs"],
    )

    faithfulness_score = faithfulness(
        generated_answer,
        result["retrieved_docs"],
    )

    return {
        "query": query,
        "answer": generated_answer,
        "answer_relevance": answer_score,
        "context_precision": context_score,
        "faithfulness": faithfulness_score,
    }

In [31]:
evaluation = evaluate_rag("What was the company's revenue?")

evaluation

{'query': "What was the company's revenue?",
 'answer': '. Opening Stock Finished goods 108.21 126.67 (c) Reconciling the amount of revenue recognised in the Statement of Profit and Loss with the contracted price : Stock-in-trade 375.94 367.69 For the year ended For the year ended Work-in-progress 17.76 15.28 March 31, 2025 March 31, 2024 Less : Closing Stock Revenue as per contracted price 6,536.15 5,974.49 Finished goods (119.16) (108.21) Add/(Less) : Adjustments Stock-in-trade (543.00) (375.94) - Sales Return (154.11) (168.31) Work-in-progress',
 'answer_relevance': 0.5677492618560791,
 'context_precision': 0.5222920298576355,
 'faithfulness': 0.840915858745575}

In [32]:
evaluation_results = []

for question in evaluation_df["question"]:

    evaluation_results.append(evaluate_rag(question))

In [33]:
evaluation_results = []

for question in evaluation_df["question"]:

    evaluation_results.append(evaluate_rag(question))

In [34]:
evaluation_results_df = pd.DataFrame(evaluation_results)

evaluation_results_df

,query,answer,answer_relevance,context_precision,faithfulness
0,What was the company's revenue?,. Opening Stock Finished goods 108.21 126.67 (...,0.567749,0.522292,0.840916
1,What was the net profit?,473 PROFIT AFTER TAX 255 277 401 450 593 691 7...,0.525217,0.491353,0.726309
2,What was the CEO salary?,. 9. In case of the demise of Mr. Rajendran du...,0.435708,0.415778,0.807415
